# Introduction

bla bla bla

# Preparation and extraction of data

In [28]:
import pandas as pd
import alive_progress

In [29]:
FILE_PATHS = [
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2024.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2023.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2022.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2021.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2020.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2019.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2018.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2017.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2016.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2015.xlsx',
    'Datasets/Raw/Erasmus-KA1-Mobility-Data-2014.xlsx',
]

In [30]:
def colCleanse(dataFrame, colList:list):
    #REQUIRES: dataFrame, column list as input
    #MODIFIES: input
    #EFFECTS: drops the specified columns

    dataFrame = dataFrame.drop(colList, axis=1)
    
    return dataFrame

As the memory taken by the dataset is sufficiently low, there is no need to actually chunk it down in order to process it and I can simply load the datasets in memory and concatenate them

Now, we can save the dataset corresponding only to university students

In [31]:
completeDF = pd.DataFrame()

removeCol = [
        'Mobility Start Month', 
        'Activity (mob)',
        'Field',
        'Participant Profile',
        'Mobility Start Year/Month',
        'Actual Participants',
        'Actual Participants ',
        'Project Reference',
        'Actual Participants (Contracted Projects)'
        ]

In [32]:
with alive_progress.alive_bar(len(FILE_PATHS), force_tty=True) as bar:
    for dataset in FILE_PATHS:
        df = pd.read_excel(dataset)
        #First thing, we should check the name of the columns and make them consistent across all datasets
        if 'Mobility Duration - calendar days' in df.columns:
            df.rename(columns={'Mobility Duration - calendar days': 'Mobility Duration'}, inplace=True)
        
        if 'Sending Organisation' in df.columns:
            df.rename(columns={'Sending Organisation': 'Sending Organization'}, inplace=True)

        if 'Receiving Organisation' in df.columns:
            df.rename(columns={'Receiving Organisation': 'Receiving Organization' }, inplace=True)
        
        # We are only interested in partecipants who are exchanging students in different universities, as such we will remove all other partecipants from the dataset
        filteredDF = df[df["Activity (mob)"].isin(['HE-LM-SMS - Student mobility for studies', 
                                                   'HE-SMS - Student mobility for studies', 
                                                   'HE-SMT - Student mobility for traineeships', 
                                                   'HE-LM-SMT - Student mobility for traineeships', 
                                                   'Student mobility for studies between Programme Countries',
                                                   'Student mobility for traineeships between Programme Countries'
                                                ])]
 
        completeDF = pd.concat([completeDF, filteredDF])
        # Save some memory
        del df, filteredDF
        #Move the animated progress bar
        bar()

completeDF.memory_usage(index=True).sum()

|████████████████████████████████████████| 11/11 [100%] in 5:55.0 (0.03/s)      ▅ 0/11 [0%] in 14s (~0s, 0.0/s)  ▃▅▇ 0/11 [0%] in 15s (~0s, 0.0/s)  ▂▂▄ 0/11 [0%] in 21s (~0s, 0.0/s)  ▂▂▄ 0/11 [0%] in 25s (~0s, 0.0/s)  █▆▄ 0/11 [0%] in 31s (~0s, 0.0/s)  ▄▆█ 0/11 [0%] in 34s (~0s, 0.0/s)  ▂▄▆ 0/11 [0%] in 36s (~0s, 0.0/s)  ▆█▆ 0/11 [0%] in 44s (~0s, 0.0/s)  ▂▄▆ 0/11 [0%] in 45s (~0s, 0.0/s) 


np.int64(609211392)

In [33]:
print(completeDF.columns)
print(completeDF.shape)

Index(['Academic Year', 'Mobility Start Month', 'Mobility Duration',
       'Activity (mob)', 'Field', 'Field of Education', 'Participant Country',
       'Education Level', 'Participant Gender', 'Participant Profile',
       'Fewer Opportunities', 'Participant Age', 'Sending Country',
       'Sending City', 'Sending Organization', 'Receiving Country',
       'Receiving City', 'Receiving Organization',
       'Actual Participants (Contracted Projects)', 'Project Reference',
       'Mobility Start Year/Month', 'Actual Participants',
       'Actual Participants '],
      dtype='str')
(3172976, 23)


In [34]:
filteredDF = colCleanse(completeDF, removeCol)
filteredDF.drop_duplicates()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
1358,2023-24,61,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,DE - Germany,Munich,-
1359,2023-24,101,0311 - Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,AT - Austria,WIEN,WU,FR - France,Toulouse,Université Toulouse Capitole
1360,2023-24,121,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1361,2023-24,171,"0410 - Business and administration, not furthe...",Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1365,2023-24,86,"0410 - Business and administration, not furthe...",Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,AT - Austria,WIEN,WU,DE - Germany,Hamburg,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215516,2014-2015,212,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Denain,College Bayard
215517,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Madrid,I.E.S. Isabel La Catolica
215518,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Alen�on,Lycee Alain
215519,2014-2015,261,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,No,18,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Mostoles,CEIP CELSO EMILIO FERREIRO


In [39]:
print(filteredDF.isnull().sum())

Academic Year             0
Mobility Duration         0
Field of Education        0
Participant Country       0
Education Level           0
Participant Gender        0
Fewer Opportunities       0
Participant Age           0
Sending Country           0
Sending City              0
Sending Organization      0
Receiving Country         0
Receiving City            0
Receiving Organization    0
dtype: int64


In [36]:
filteredDF = filteredDF.dropna()
print(filteredDF.shape)

(3172958, 14)


As our work focuses on university students, we can safely drop some columns:
- Moblity  Start Month & Mobility Start Year/Month: Is it of any use to our analysis? Also contains high number of NaNs
- Activity : As it has already been used for filtering, we can safely drop
- Field : As it is no longer useful after the filtering has been completed
- Partecipant Profile: All of these will be Learners once we are done with the filtering
- Age: Could maybe do something with this? 
- Actual Participants is the last column of most datasets, I don't really know wheter it can be useful for us
-      'Project Reference',
       'Mobility Start Year/Month', 'Actual Participants',
       'Actual Participants ', ,
        'Project Reference'
are all column I...quite frankly do not know where they come from, I've checked all Datasets but none of them contain such columns, and all I did was stack them with concat. In any case, let's trim them and check later if the result is coherent. They are shown to be from 2015, but I have not found them in the 2015 Dataset
- 'Mobility Duration - calendar days' should be reconducted to Mobility Duration
- 'Sending Organisation', 'Receiving Organisation', should be reconducted to their Organi**z**ation variant

As over half of the values in Actual Participants is NaN, I'd drop the column. Mobility Duration, Sending Organization and Receiving Organization's NaN could be related, look into the Dataset

Note varie

Mentre controllavo le colonne, ho visto che alcune che hanno alto numero di NaN sono semplicemente altre forme grammaticali di colonne esistenti in altre versioni del Dataset. Chiunque abbia fatto questa cosa è un cane, ma devo riconciliare le versioni.
Oltre a questo, resta il problema di ricondurre i corsi ai dipartimenti o, quantomeno, a una versione unica dei corsi piuttosto che ad annotazioni strambe del medesimo corso.


Dal 2014 fino a ___ abbiamo le colonne: 
- Organization scritta come Organi**s**ation
- 

Check the columns names and, if a specific one is present, rename it in the correct format. 

In [40]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):  
    display(filteredDF['Field of Education'].unique())

lista = filteredDF['Field of Education'].unique()

<StringArray>
[                       'Business and administration',
                                          'Economics',
                   'Business, administration and law',
                              'Mining and extraction',
                         'Mechanics and metal trades',
                             'Electricity and energy',
                             'Environmental sciences',
                 'Engineering and engineering trades',
                         'Electronics and automation',
                          'Music and performing arts',
 ...
 'Traditional and complementary medicine and therapy',
                                          'Fisheries',
                                                  '-',
                                           'Services',
           'Hygiene and occupational health services',
                     'Occupational health and safety',
                                        'Work skills',
                                  'Domestic se

We will now fix the column names for the courses

In [38]:
import json


with open('mapping_results.json', 'r') as map:
    mapping_data = json.load(map)

lookup_map = mapping_data.get('flat_lookup', {})

filteredDF['Field of Education'] = filteredDF['Field of Education'].map(lookup_map).fillna(filteredDF['Field of Education'])

display(filteredDF)

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
1358,2023-24,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,DE - Germany,Munich,-
1359,2023-24,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,AT - Austria,WIEN,WU,FR - France,Toulouse,Université Toulouse Capitole
1360,2023-24,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1361,2023-24,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,AT - Austria,WIEN,WU,FR - France,Paris,-
1365,2023-24,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,AT - Austria,WIEN,WU,DE - Germany,Hamburg,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215516,2014-2015,212,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Denain,College Bayard
215517,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Madrid,I.E.S. Isabel La Catolica
215518,2014-2015,243,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,No,19,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,FR - France,Alen�on,Lycee Alain
215519,2014-2015,261,Languages,United Kingdom,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,No,18,UK - United Kingdom,OXFORD,OXFORD BROOKES UNIVERSITY,ES - Spain,Mostoles,CEIP CELSO EMILIO FERREIRO


In [ ]:
filteredDF.to_csv('Datasets/Processed/Erasmus-Data.csv')